# INF-CONTINUOUS-BATCHING. Continuous Batching | 连续批处理

**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `连续批处理`, `迭代调度`

## 本节导读

静态批处理会等待同批请求全部完成，较短请求留下的 slot 因而闲置。连续批处理在每轮前向结束后移出终止请求，再依据 slot、KV 容量和本轮 Token 预算接纳新请求。你将沿着一次迭代边界，理解批次成员如何变化，以及吞吐、公平性与尾延迟为什么会互相制约。

## 前置阅读

- [INF-ITERATION-BUDGET：Serving 迭代成本与批次容量上界](./INF-ITERATION-BUDGET_Serving_Iteration_Cost_and_Batch_Bounds.ipynb)：单轮成本与五类上界。
- [36：Decode Scheduling](./36_Decode_Scheduling.ipynb)：请求级优先顺序。
- [37：KV Cache 生命周期与容量控制](./37_KV_Cache_Lifecycle_and_Capacity_Control.ipynb)：准入所需的可用容量状态。

### Step 1：批次为什么要在迭代边界重组

自回归 Decode 每轮通常只为每个活跃请求生成一个 token。请求完成时间不同，固定批次会逐渐出现空槽；连续批处理在迭代边界释放终止成员，并从等待队列补入新请求。请求排序、批次重组和 KV 容量分别提供优先级、执行成员与资源约束，三者共同决定下一轮实际批次。

| 调度对象 | 提供的信息 | 对本轮批次的作用 |
|:---|:---|:---|
| 请求级调度 | 候选请求的优先顺序 | 决定等待队列的检查次序 |
| 连续批处理 | 活跃成员、空闲 slot、迭代边界 | 决定本轮实际执行成员 |
| KV 容量控制 | 可用 block 与请求状态量 | 决定候选是否具备准入条件 |

![连续批处理的迭代循环](https://github.com/yanghaocsg/llm-algo-leetcode/blob/main/docs/public/02_PyTorch_Algorithms/inf_continuous_batching_overview.svg?raw=1)


### Step 2：slot、KV 与本轮 token 预算共同限制准入

空 slot 只是必要条件。新请求还要满足 KV block 预算，并能把本轮需要执行的 token 放进 `max_num_batched_tokens`：Decode 请求通常占 1 个 scheduled token，Prefill 请求则占一个 chunk。任一预算不足时，请求继续排队，不能仅因为 slot 空闲就强行接纳。

| 状态 | 进入本轮 batch 的条件 | 状态变化 |
|:---|:---|:---|
| active | 尚未完成且 KV 状态有效 | 执行一个 token 后更新剩余长度 |
| finished | 达到 EOS 或 token 上限 | 退出 batch，释放 slot 与引用 |
| waiting | 优先级满足且 slot / KV / token budget 均可用 | 转为 active |
| blocked | 任一预算不足 | 保留在等待队列 |

![Static Batch 与 Continuous Batch 的 slot 使用差异](https://github.com/yanghaocsg/llm-algo-leetcode/blob/main/docs/public/02_PyTorch_Algorithms/inf_continuous_batching_slots.svg?raw=1)

### Step 3：吞吐、公平性与尾延迟如何冲突

持续补满 batch 往往提高 token throughput，却可能让长请求或低优先级请求持续等待。真实调度器还要限制最大 batch tokens、Prefill 工作量和单请求等待时间，并记录 queue wait、batch occupancy、P95/P99 与 starvation。

| 策略偏向 | 可能收益 | 需要监控的风险 |
|:---|:---|:---|
| 始终补满 slot | 吞吐与利用率 | 新请求 TTFT、长请求干扰 |
| 只选短请求 | 平均延迟 | 长请求 starvation |
| aging / deadline | 公平性与 SLO | batch 效率下降 |
| 严格 KV gate | 避免运行中 OOM | 准入不足、设备空闲 |

### Step 4：实现一次迭代边界的批次重组

本题用活跃请求、等待队列和三类预算复现一次迭代边界。请求字典、队列遍历、账本更新和返回结构已经给出；你将补全阶段工作量、终止状态判断、准入阻塞原因和状态推进。测试分别覆盖 Prefill / Decode、终止回收、三重资源门槛与阶段迁移。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1 · `iteration_token_demand` | 计算本轮 Token 需求 | Prefill 不超过剩余量与 chunk；Decode 最多推进 1 Token | 长短 Prefill、Decode 与未知阶段 |
| TODO 2 · `release_terminal_requests` | 识别迭代边界需要释放的请求 | 终止请求移出 active 并释放其 KV block | 活跃与完成请求混合 |
| TODO 3 · `form_iteration_batch` | 按资源门槛组成本轮批次 | 按 slot → KV → Token 顺序返回首个阻塞原因；不跳过队首 | 三类阻塞、成功准入与队首保留 |
| TODO 4 · `advance_iteration` | 推进请求剩余工作量 | Prefill 归零后转 Decode；Decode 归零后转 completed | 两类阶段迁移与计数归零 |

In [4]:
from copy import deepcopy

In [59]:
TERMINAL_STATES = {'completed', 'cancelled', 'timed_out', 'failed'}


def iteration_token_demand(request, prefill_chunk_size):
    """返回请求在当前迭代需要处理的 token 数。"""
    # {'id': request_id, 'phase': phase, 'remaining_prefill_tokens': prefill, 'remaining_decode_tokens': decode, 'kv_blocks': blocks, 'status': status}
    # iteration_token_demand(('p', 'prefill', 10, 4, 2), 4) == 4
    # iteration_token_demand(_request('p', 'prefill', 2, 4, 2), 4) == 2
    # iteration_token_demand(_request('d', 'decode', 0, 3, 2), 4) == 1
    if prefill_chunk_size <= 0:
        raise ValueError('prefill_chunk_size 必须为正数')
    # TODO 1（阶段工作量）：分别计算 Prefill 与 Decode 的本轮需求。
    # prefill_demand = ???  # 不超过剩余 Prefill token 和 chunk 上限。
    # decode_demand = ???  # 仍有 Decode 工作时为 1，否则为 0。
    prefill_demand = min(max(request['remaining_prefill_tokens'], 0), prefill_chunk_size)
    decode_demand = 1 if request['remaining_decode_tokens'] > 0 else 0
    if request['phase'] == 'prefill':
        return prefill_demand
    if request['phase'] == 'decode':
        return decode_demand
    raise ValueError('未知请求阶段')


def release_terminal_requests(active_requests):
    """返回仍活跃请求、终止请求和释放的 KV blocks。"""
    active, terminal = [], []
    for request in active_requests:
        # TODO 2（边界释放）：判断当前请求是否已经进入终止状态。
        # is_terminal = ???  # 仅 completed/cancelled/timed_out/failed 为真。
      is_terminal = 1 if request['status'] in ['completed', 'cancelled', 'timed_out', 'failed'] else 0
      if is_terminal:
        terminal.append(request)
      else:
        active.append(request)
      released_blocks = sum(item['kv_blocks'] for item in terminal)
    return active, terminal, released_blocks


def form_iteration_batch(active_requests, waiting_requests, max_batch_size, free_kv_blocks, max_num_batched_tokens, prefill_chunk_size):
    """在 slot、KV 与 token 预算下形成本轮批次。"""
    active, terminal, released = release_terminal_requests(active_requests)
    available_blocks = free_kv_blocks + released
    free_slots = max_batch_size - len(active)
    batch = []
    for request in active:
        item = deepcopy(request)
        item['scheduled_tokens'] = iteration_token_demand(item, prefill_chunk_size)
        batch.append(item)
    free_tokens = max_num_batched_tokens - sum(item['scheduled_tokens'] for item in batch)
    if free_slots < 0 or free_tokens < 0:
        raise ValueError('active 请求已经超过本轮预算')

    waiting, blocked = deepcopy(waiting_requests), []
    admitted_count = 0
    while admitted_count < len(waiting):
        request = waiting[admitted_count]
        demand = iteration_token_demand(request, prefill_chunk_size)
        # TODO 3（准入门槛）：按 slot、KV、token 的次序确定首个阻塞原因。
        # block_reason = ???  # 可准入时为 None，否则为 slot/kv_blocks/token_budget。
        print('demand', request['phase'], demand)
        print('free_slots', free_slots, free_kv_blocks, free_tokens)
        block_reason = None
        if free_slots <= 0:
          block_reason = 'slot'
          blocked.append({'id':request['id'], 'reason':'slot'})
        elif free_kv_blocks < request['kv_blocks']:
          block_reason = 'kv_blocks'
          blocked.append({'id':request['id'], 'reason':'kv_blocks'})
        elif free_tokens < demand:
          block_reason = 'token_budget'
          blocked.append({'id':request['id'], 'reason':'token_budget'})
        if block_reason:
          print('blocked', blocked)
          break
        item = deepcopy(request)
        item['scheduled_tokens'] = demand
        item['status'] = 'active'
        batch.append(item)
        print('batch', batch)
        free_slots -= 1
        available_blocks -= item['kv_blocks']
        free_tokens -= demand
        admitted_count += 1
    return {
        'batch': batch, 'waiting': waiting[admitted_count:], 'terminal': terminal,
        'blocked': blocked, 'released_kv_blocks': released, 'free_slots': free_slots,
        'free_kv_blocks': available_blocks, 'free_token_budget': free_tokens,
    }


def advance_iteration(batch):
    """根据 scheduled tokens 推进 Prefill/Decode 状态。"""
    updated = []
    for request in deepcopy(batch):
        amount = request['scheduled_tokens']
        # TODO 4（状态推进）：从对应阶段的剩余工作量中扣除本轮 token。
        # remaining_tokens = ???  # 当前阶段执行后的剩余 token，不能为负。
        #demand = iteration_token_demand(request)
        demand = request['remaining_prefill_tokens'] if request['phase']=='prefill' else request['remaining_decode_tokens']
        remaining_tokens = amount - demand
        if remaining_tokens < 0:
            raise ValueError('scheduled_tokens 不能超过阶段剩余工作量')
        if request['phase'] == 'prefill':
            request['remaining_prefill_tokens'] = remaining_tokens
            if remaining_tokens == 0:
                request['phase'] = 'decode'
        elif request['phase'] == 'decode':
            request['remaining_decode_tokens'] = remaining_tokens
            if remaining_tokens == 0:
                request['status'] = 'completed'
        else:
            raise ValueError('未知请求阶段')
        updated.append(request)
    return updated


In [60]:
def _request(request_id, phase, prefill, decode, blocks, status='active'):
    """构造最小请求状态，供各机制测试复用。"""
    return {'id': request_id, 'phase': phase, 'remaining_prefill_tokens': prefill, 'remaining_decode_tokens': decode, 'kv_blocks': blocks, 'status': status}
def test_phase_token_demand():
    """Prefill 受 chunk 限制，Decode 每轮最多推进一个 token。"""
    assert iteration_token_demand(_request('p', 'prefill', 10, 4, 2), 4) == 4
    assert iteration_token_demand(_request('p', 'prefill', 2, 4, 2), 4) == 2
    assert iteration_token_demand(_request('d', 'decode', 0, 3, 2), 4) == 1

print(iteration_token_demand(_request('p', 'prefill', 10, 4, 2), 4))
test_phase_token_demand()

def test_batch_gates_and_block_reason():
    """队首请求未通过资源门槛时应保留队列并报告原因。"""
    active = [_request('a', 'decode', 0, 2, 2)]
    waiting = [_request('p', 'prefill', 6, 3, 2), _request('d', 'decode', 0, 3, 1)]
    report = form_iteration_batch(active, waiting, 3, 4, 4, 4)
    print('report', report['batch'])
    print('block', report['blocked'], report['waiting'])
    assert [x['id'] for x in report['batch']] == ['a']
    assert report['blocked'][0] == {'id': 'p', 'reason': 'token_budget'}
    assert [x['id'] for x in report['waiting']] == ['p', 'd']
    assert report['free_token_budget'] == 3

#test_batch_gates_and_block_reason()
def test_iteration_state_advance():
    """执行后应正确推进 Prefill→Decode 与 Decode→completed。"""
    prefill = _request('p', 'prefill', 3, 2, 2); prefill['scheduled_tokens'] = 3
    decode = _request('d', 'decode', 0, 1, 2); decode['scheduled_tokens'] = 1
    print('prefill', prefill, 'decode', decode)
    updated = advance_iteration([prefill, decode])
    assert updated[0]['phase'] == 'decode' and updated[0]['remaining_prefill_tokens'] == 0
    assert updated[1]['status'] == 'completed' and updated[1]['remaining_decode_tokens'] == 0
test_iteration_state_advance()

4
prefill {'id': 'p', 'phase': 'prefill', 'remaining_prefill_tokens': 3, 'remaining_decode_tokens': 2, 'kv_blocks': 2, 'status': 'active', 'scheduled_tokens': 3} decode {'id': 'd', 'phase': 'decode', 'remaining_prefill_tokens': 0, 'remaining_decode_tokens': 1, 'kv_blocks': 2, 'status': 'active', 'scheduled_tokens': 1}


### 测试

测试分别验证阶段工作量、终止释放、三重资源 gate、队首阻塞原因和一轮状态推进。

In [61]:
def _request(request_id, phase, prefill, decode, blocks, status='active'):
    """构造最小请求状态，供各机制测试复用。"""
    return {'id': request_id, 'phase': phase, 'remaining_prefill_tokens': prefill, 'remaining_decode_tokens': decode, 'kv_blocks': blocks, 'status': status}

def test_phase_token_demand():
    """Prefill 受 chunk 限制，Decode 每轮最多推进一个 token。"""
    assert iteration_token_demand(_request('p', 'prefill', 10, 4, 2), 4) == 4
    assert iteration_token_demand(_request('p', 'prefill', 2, 4, 2), 4) == 2
    assert iteration_token_demand(_request('d', 'decode', 0, 3, 2), 4) == 1

def test_terminal_release():
    """终止请求应离开 active 并释放其 KV block。"""
    active = [_request('a', 'decode', 0, 2, 3), _request('b', 'decode', 0, 0, 4, 'completed')]
    kept, terminal, released = release_terminal_requests(active)
    assert [x['id'] for x in kept] == ['a'] and [x['id'] for x in terminal] == ['b']
    assert released == 4

def test_batch_gates_and_block_reason():
    """队首请求未通过资源门槛时应保留队列并报告原因。"""
    active = [_request('a', 'decode', 0, 2, 2)]
    waiting = [_request('p', 'prefill', 6, 3, 2), _request('d', 'decode', 0, 3, 1)]
    report = form_iteration_batch(active, waiting, 3, 4, 4, 4)
    assert [x['id'] for x in report['batch']] == ['a']
    assert report['blocked'][0] == {'id': 'p', 'reason': 'token_budget'}
    assert [x['id'] for x in report['waiting']] == ['p', 'd']
    assert report['free_token_budget'] == 3

def test_iteration_state_advance():
    """执行后应正确推进 Prefill→Decode 与 Decode→completed。"""
    prefill = _request('p', 'prefill', 3, 2, 2); prefill['scheduled_tokens'] = 3
    decode = _request('d', 'decode', 0, 1, 2); decode['scheduled_tokens'] = 1
    updated = advance_iteration([prefill, decode])
    assert updated[0]['phase'] == 'decode' and updated[0]['remaining_prefill_tokens'] == 0
    assert updated[1]['status'] == 'completed' and updated[1]['remaining_decode_tokens'] == 0

test_phase_token_demand(); test_terminal_release(); test_batch_gates_and_block_reason(); test_iteration_state_advance()
print('✅ Continuous Batching 机制测试通过')

demand prefill 4
free_slots 2 4 3
blocked [{'id': 'p', 'reason': 'token_budget'}]
✅ Continuous Batching 机制测试通过


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难，再向下查看参考代码与解析。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码

In [ ]:
TERMINAL_STATES = {'completed', 'cancelled', 'timed_out', 'failed'}


# TODO 1：计算本轮 Token 需求。
def iteration_token_demand(request, prefill_chunk_size):
    """返回请求在当前迭代需要处理的 token 数。"""
    if prefill_chunk_size <= 0:
        raise ValueError('prefill_chunk_size 必须为正数')
    prefill_demand = min(max(request['remaining_prefill_tokens'], 0), prefill_chunk_size)
    decode_demand = 1 if request['remaining_decode_tokens'] > 0 else 0
    if request['phase'] == 'prefill':
        return prefill_demand
    if request['phase'] == 'decode':
        return decode_demand
    raise ValueError('未知请求阶段')


# TODO 2：识别迭代边界需要释放的请求。
def release_terminal_requests(active_requests):
    """返回仍活跃请求、终止请求和释放的 KV blocks。"""
    active, terminal = [], []
    for request in active_requests:
        is_terminal = request['status'] in TERMINAL_STATES
        target = terminal if is_terminal else active
        target.append(deepcopy(request))
    released_blocks = sum(item['kv_blocks'] for item in terminal)
    return active, terminal, released_blocks


# TODO 3：按资源门槛组成本轮批次。
def form_iteration_batch(active_requests, waiting_requests, max_batch_size, free_kv_blocks, max_num_batched_tokens, prefill_chunk_size):
    """在 slot、KV 与 token 预算下形成本轮批次。"""
    active, terminal, released = release_terminal_requests(active_requests)
    available_blocks = free_kv_blocks + released
    free_slots = max_batch_size - len(active)
    batch = []
    for request in active:
        item = deepcopy(request)
        item['scheduled_tokens'] = iteration_token_demand(item, prefill_chunk_size)
        batch.append(item)
    free_tokens = max_num_batched_tokens - sum(item['scheduled_tokens'] for item in batch)
    if free_slots < 0 or free_tokens < 0:
        raise ValueError('active 请求已经超过本轮预算')
    waiting, blocked = deepcopy(waiting_requests), []
    admitted_count = 0
    while admitted_count < len(waiting):
        request = waiting[admitted_count]
        demand = iteration_token_demand(request, prefill_chunk_size)
        block_reason = (
            'slot' if free_slots <= 0 else
            'kv_blocks' if request['kv_blocks'] > available_blocks else
            'token_budget' if demand > free_tokens else None
        )
        if block_reason is not None:
            blocked.append({'id': request['id'], 'reason': block_reason})
            break
        item = deepcopy(request)
        item['scheduled_tokens'] = demand
        item['status'] = 'active'
        batch.append(item)
        free_slots -= 1
        available_blocks -= item['kv_blocks']
        free_tokens -= demand
        admitted_count += 1
    return {
        'batch': batch, 'waiting': waiting[admitted_count:], 'terminal': terminal,
        'blocked': blocked, 'released_kv_blocks': released, 'free_slots': free_slots,
        'free_kv_blocks': available_blocks, 'free_token_budget': free_tokens,
    }


# TODO 4：推进请求剩余工作量。
def advance_iteration(batch):
    """根据 scheduled tokens 推进 Prefill/Decode 状态。"""
    updated = []
    for request in deepcopy(batch):
        amount = request['scheduled_tokens']
        key = 'remaining_prefill_tokens' if request['phase'] == 'prefill' else 'remaining_decode_tokens'
        remaining_tokens = request[key] - amount
        if remaining_tokens < 0:
            raise ValueError('scheduled_tokens 不能超过阶段剩余工作量')
        if request['phase'] == 'prefill':
            request['remaining_prefill_tokens'] = remaining_tokens
            if remaining_tokens == 0:
                request['phase'] = 'decode'
        elif request['phase'] == 'decode':
            request['remaining_decode_tokens'] = remaining_tokens
            if remaining_tokens == 0:
                request['status'] = 'completed'
        else:
            raise ValueError('未知请求阶段')
        updated.append(request)
    return updated


### 解析

#### TODO 1：计算阶段工作量

- **机制**：预填充工作量受剩余输入和块上限共同约束，解码请求每轮推进一个 token。
- **不变量**：本轮需求不超过当前阶段的剩余工作量。
- **测试证据**：长短预填充与解码样例验证阶段边界。

#### TODO 2：释放终止请求

- **机制**：在迭代边界移除终止请求，并把它占用的 KV 块归还容量池。
- **不变量**：未终止请求保留在活跃集合，释放量只来自本轮终止成员。
- **测试证据**：混合请求测试同时核对保留成员、终止成员和释放容量。

#### TODO 3：判断准入门槛

- **机制**：候选依次检查批次槽位、KV 容量和本轮 token 预算。
- **不变量**：保留首个阻塞原因，队首请求不能被静默丢弃或被后续成员绕过。
- **测试证据**：测试分别制造三类资源不足并核对阻塞原因。

#### TODO 4：推进请求状态

- **机制**：从当前阶段扣除本轮执行量，输入处理完毕后进入解码，输出完成后进入终止状态。
- **不变量**：阶段切换只发生在剩余工作量归零时，计数不得为负。
- **测试证据**：测试核对阶段切换、终止状态和剩余计数。

### Step 5：真实后端实验——批次屏障与滚动补位

CPU 题目验证迭代边界上的释放、准入和状态推进；真实后端固定同一服务、请求清单和并发槽位，只改变客户端准入方式，比较等待整批完成与空槽立即补位。该对照验证准入边界，不替换 vLLM 内部调度器。

| 学习问题 | CPU 已确认 | 真实后端要确认 |
|---|---|---|
| 请求何时退出与补位 | 终止请求释放槽位与 KV 容量 | 滚动补位是否减少空闲槽位时间 |
| 等待发生在哪里 | 准入状态与迭代状态分离 | 逻辑就绪到实际提交的等待是否下降 |
| 吞吐收益是否免费 | 状态机与预算守恒 | 吞吐改善是否伴随尾延迟恶化 |

#### 5.1 环境与固定实验条件

本实验使用 `INF-VLLM-011-CU128`。首次运行先完成[推理服务环境 Quickstart](../topic_discussion/inference_optimization/quickstart.md#inf-vllm-quickstart)，启动固定 vLLM 服务并确认 `/v1/models` 可访问；Notebook 不安装依赖，也不启动或停止服务。本页不替换 vLLM 内部调度器，而是在同一服务上分别模拟静态批次屏障和完成即补位，观察请求提交边界带来的差异。

| 配置项 | 固定值 | 作用 |
|---|---|---|
| 模型与服务 | `Qwen/Qwen3-0.6B`，vLLM 0.11 | 保持执行后端一致 |
| 请求清单 | 12 条固定 128-token 输入，四档强制输出长度 | 制造可复现的批内完成时间差 |
| 并发度 | 4 | 两种提交方式使用同一槽位上限 |
| 对照变量 | 批次屏障 / 滚动补位 | 只改变完成后的补位时机 |


In [ ]:
# 5.1 只保留学习者需要修改的运行开关与结果标签。
RUN_MODE = "dry_run"  # 默认值 dry_run：只登记计划；服务就绪后改为 real_backend。
RUN_LABEL = "default"  # 默认值 default：正式复测改为唯一标签，避免覆盖历史结果。
RUN_ORDER = "barrier_then_rolling"  # 默认值 barrier_then_rolling：交换顺序复测时改为 rolling_then_barrier。

ENVIRONMENT_PROFILE = "INF-VLLM-011-CU128"  # Quickstart 中登记的独立服务环境。
BACKEND_BASE_URL = "http://127.0.0.1:8000"  # 已启动的 OpenAI-compatible 服务地址。
MODEL_ID = "Qwen/Qwen3-0.6B"  # 两组实验使用同一个服务模型。
MODEL_REVISION = "main"  # 5.2 优先从本地缓存解析实际 commit。
PROMPT_TOKENS = 128  # 默认值 128：两种准入方式固定相同输入长度。
CONCURRENCY = 4  # 两种提交方式共用的最大并发请求数。
WARMUP_STEPS = 1  # 每种提交方式开始前的预热请求数。
TIMEOUT_SECONDS = 180.0  # 单请求超时上限。
OUTPUT_RELATIVE_PATH = f"02_PyTorch_Algorithms/benchmarks/results/inf_continuous_batching/continuous_batching_{RUN_LABEL}.json"


#### 5.2 执行固定请求清单的配对实验

两组请求使用相同输入 token 数、强制输出长度、顺序和并发度。批次屏障必须等待当前四条请求全部结束才提交下一组；滚动补位在任一请求结束后立即提交下一条。所有请求在实验开始时均视为逻辑就绪，实际提交时间就是准入等待。


In [ ]:
# 5.2 执行两种到达方式并保存请求级证据。
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer
from tools.inference_project_runtime import locate_repo_root
from tools.notebook_runtime import package_version, resolve_model_commit
from tools.serving_mechanism_runtime import percentile, run_arrival_schedule

if RUN_MODE not in {"dry_run", "real_backend"}:
    raise ValueError("RUN_MODE 必须是 dry_run 或 real_backend；服务实验不能使用 real_gpu")
if RUN_ORDER not in {"barrier_then_rolling", "rolling_then_barrier"}:
    raise ValueError("RUN_ORDER 必须是 barrier_then_rolling 或 rolling_then_barrier")
root = locate_repo_root()
output_path = root / OUTPUT_RELATIVE_PATH
length_pattern = (16, 64, 32, 96)
expected_output_tokens = [length_pattern[index % len(length_pattern)] for index in range(12)]
record = {
    "schema_version": "inference-continuous-batching/v2",
    "semantic_id": "INF-CONTINUOUS-BATCHING",
    "environment_profile": ENVIRONMENT_PROFILE,
    "run_mode": RUN_MODE,
    "run_label": RUN_LABEL,
    "framework": {"torch": torch.__version__, "vllm": package_version("vllm")},
    "workload": {
        "id": "inf-continuous-batching-arrival-v2",
        "model_id": MODEL_ID,
        "requested_revision": MODEL_REVISION,
        "resolved_commit": None,
        "request_count": len(expected_output_tokens),
        "prompt_tokens": PROMPT_TOKENS,
        "expected_output_tokens": expected_output_tokens,
        "concurrency": CONCURRENCY,
        "run_order": RUN_ORDER,
    },
    "baseline": None, "candidate": None, "comparison": None,
    "failure": None, "evidence_level": "dry_run", "decision": "not_measured",
}

if RUN_MODE == "real_backend":
    try:
        commit, source = resolve_model_commit(MODEL_ID, MODEL_REVISION, local_files_only=True)
        record["workload"]["resolved_commit"] = commit
        record["workload"]["revision_source"] = source
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=commit, local_files_only=True)
        def fixed_token_prompt(index, target_tokens):
            suffix = f"\n请求编号 {index}：解释连续批处理的状态变化。"
            suffix_ids = tokenizer(suffix, add_special_tokens=False)["input_ids"]
            if len(suffix_ids) >= target_tokens:
                raise ValueError("PROMPT_TOKENS 必须大于请求后缀长度")
            seed = f"样本 {index} 的调度槽位、请求状态与完成边界。"
            body_ids = tokenizer(seed * (target_tokens // 4 + 16), add_special_tokens=False)["input_ids"]
            body_ids = body_ids[:target_tokens - len(suffix_ids)]
            return tokenizer.decode(body_ids + suffix_ids, skip_special_tokens=True)
        requests = [
            {
                "prompt": fixed_token_prompt(index, PROMPT_TOKENS),
                "max_tokens": output_tokens,
                "request_role": "scheduled_request",
                "ignore_eos": True,
            }
            for index, output_tokens in enumerate(expected_output_tokens)
        ]
        record["workload"]["client_content_token_counts"] = [
            len(tokenizer(item["prompt"], add_special_tokens=False)["input_ids"]) for item in requests
        ]
        cases = {"baseline": "barrier_waves", "candidate": "rolling_admission"}
        order = ["baseline", "candidate"]
        if RUN_ORDER == "rolling_then_barrier":
            order.reverse()
        for role in order:
            record[role] = run_arrival_schedule(
                base_url=BACKEND_BASE_URL, model=MODEL_ID, prompts=requests,
                schedule=cases[role], concurrency=CONCURRENCY, max_tokens=max(expected_output_tokens),
                warmup=WARMUP_STEPS, timeout_s=TIMEOUT_SECONDS,
            )
        for report in (record["baseline"], record["candidate"]):
            successful = [item for item in report["requests"] if item.get("ok")]
            ordered = sorted(successful, key=lambda item: item["request_id"])
            workload_ttft = [item["submitted_ms"] + item["ttft_ms"] for item in successful]
            workload_e2e = [item["completed_ms"] for item in successful]
            admission_wait = [item["submitted_ms"] for item in successful]
            makespan_ms = max(workload_e2e, default=0.0)
            occupied_slot_ms = sum(item["completed_ms"] - item["submitted_ms"] for item in successful)
            available_slot_ms = CONCURRENCY * makespan_ms
            events = []
            for item in successful:
                item["logical_ready_ms"] = 0.0
                item["admission_wait_ms"] = item["submitted_ms"]
                events.extend([
                    (item["submitted_ms"], 1, item["request_id"]),
                    (item["completed_ms"], -1, item["request_id"]),
                ])
            active, timeline = 0, []
            for timestamp_ms, delta, request_id in sorted(events, key=lambda event: (event[0], event[1])):
                active += delta
                timeline.append({"timestamp_ms": timestamp_ms, "active_requests": active, "request_id": request_id})
            report["slot_timeline"] = timeline
            report["metrics"]["workload_ttft_ms"] = {
                "p95": percentile(workload_ttft, 95), "p99": percentile(workload_ttft, 99),
            }
            report["metrics"]["workload_e2e_ms"] = {
                "p95": percentile(workload_e2e, 95), "p99": percentile(workload_e2e, 99),
            }
            report["metrics"]["admission_wait_ms"] = {
                "p95": percentile(admission_wait, 95), "p99": percentile(admission_wait, 99),
            }
            report["metrics"]["slot_accounting"] = {
                "occupied_slot_ms": occupied_slot_ms,
                "available_slot_ms": available_slot_ms,
                "idle_slot_ms": max(0.0, available_slot_ms - occupied_slot_ms),
                "slot_utilization": occupied_slot_ms / max(available_slot_ms, 1e-12),
            }
            observed = [item["output_tokens"] for item in ordered]
            report["metrics"]["observed_output_tokens"] = observed
            report["metrics"]["output_lengths_exact"] = observed == expected_output_tokens
        baseline_metrics = record["baseline"]["metrics"]
        candidate_metrics = record["candidate"]["metrics"]
        throughput_ratio = candidate_metrics["request_throughput_per_s"] / max(baseline_metrics["request_throughput_per_s"], 1e-12)
        workload_ttft_p99_ratio = candidate_metrics["workload_ttft_ms"]["p99"] / max(baseline_metrics["workload_ttft_ms"]["p99"], 1e-12)
        workload_e2e_p99_ratio = candidate_metrics["workload_e2e_ms"]["p99"] / max(baseline_metrics["workload_e2e_ms"]["p99"], 1e-12)
        admission_wait_p99_ratio = candidate_metrics["admission_wait_ms"]["p99"] / max(baseline_metrics["admission_wait_ms"]["p99"], 1e-12)
        slot_utilization_gain = candidate_metrics["slot_accounting"]["slot_utilization"] - baseline_metrics["slot_accounting"]["slot_utilization"]
        idle_slot_ratio = candidate_metrics["slot_accounting"]["idle_slot_ms"] / max(baseline_metrics["slot_accounting"]["idle_slot_ms"], 1e-12)
        output_lengths_exact = baseline_metrics["output_lengths_exact"] and candidate_metrics["output_lengths_exact"]
        all_succeeded = baseline_metrics["failed_requests"] == candidate_metrics["failed_requests"] == 0
        record["comparison"] = {
            "request_throughput_ratio": throughput_ratio,
            "workload_ttft_p99_ratio": workload_ttft_p99_ratio,
            "workload_e2e_p99_ratio": workload_e2e_p99_ratio,
            "admission_wait_p99_ratio": admission_wait_p99_ratio,
            "slot_utilization_gain": slot_utilization_gain,
            "idle_slot_ratio": idle_slot_ratio,
            "output_lengths_exact": output_lengths_exact,
            "all_requests_succeeded": all_succeeded,
        }
        record["evidence_level"] = "real_backend_mechanism_benchmark"
        record["decision"] = "accept_rolling_admission" if all_succeeded and output_lengths_exact and throughput_ratio > 1.03 and workload_e2e_p99_ratio <= 1.05 and admission_wait_p99_ratio < 0.95 and slot_utilization_gain >= 0.05 and idle_slot_ratio < 0.90 else "tune"
    except Exception as exc:
        record.update({
            "failure": {"type": type(exc).__name__, "message": str(exc)},
            "evidence_level": "failed_real_backend_attempt",
            "decision": "reject_until_fixed",
        })

output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(record, ensure_ascii=False, indent=2))


#### 5.3 读取配对实验结果

结果必须同时包含两种提交方式的请求级记录和汇总指标；只完成服务预检不能形成连续批处理证据。


In [ ]:
# 5.3 只读取 5.2 保存的 JSON；重启 Kernel 后不依赖 5.2 的内存状态。
import json
from tools.inference_project_runtime import locate_repo_root

output_path = locate_repo_root() / OUTPUT_RELATIVE_PATH
saved = json.loads(output_path.read_text(encoding="utf-8"))
if saved["run_mode"] != "real_backend":
    summary = {"status": "dry_run", "next_action": "启动服务后将 RUN_MODE 改为 real_backend"}
elif saved["failure"] is not None:
    raise RuntimeError(saved["failure"])
else:
    summary = {
        "baseline": saved["baseline"]["metrics"],
        "candidate": saved["candidate"]["metrics"],
        "comparison": saved["comparison"],
        "decision": saved["decision"],
    }
summary


#### 5.4 解释吞吐与尾延迟

v2 固定 128-token 输入并强制执行 16/32/64/96-token 输出矩阵。正向和反向两轮均完整生成 12 条请求；滚动补位把槽位利用率提高 18.1–23.9 个百分点，空闲槽位时间减少 59.2%–68.4%，准入等待 P99 减少 44.2%–52.8%，请求吞吐提高 34.6%–40.4%。两轮均接受滚动补位；TPOT 变化方向不一致，不能据此声称解码内核变快。

| 复测与运行顺序 | 槽位利用率：屏障 → 滚动 | 空闲槽位时间比 | 准入等待 P99 比 | 吞吐比 | 工作负载 E2E P99 比 | 决策 |
|---|---:|---:|---:|---:|---:|---|
| v2 正向：屏障 → 滚动 | 0.584 → 0.823 | 0.316 | 0.558 | 1.346 | 0.735 | 接受 |
| v2 反向：滚动 → 屏障 | 0.576 → 0.757 | 0.408 | 0.472 | 1.404 | 0.699 | 接受 |


## 相关阅读

- [Orca: A Distributed Serving System for Transformer-Based Generative Models](https://www.usenix.org/conference/osdi22/presentation/yu)
- [vLLM Scheduling](https://docs.vllm.ai/en/latest/design/)
- [70：Serving Scheduler Benchmark](./70_Serving_Scheduler_Benchmark.ipynb)